# AppScout in Colab

Everything the AppScout website does, in a notebook: describe an app idea, and get its viability score, why it scored
that, how to make it a better business, its competitors and its market.

**How to use it**

1. **Runtime → Run all.** The first cell fetches the project from GitHub. Without internet access it asks you to
   upload `AppScout.zip` instead.
2. Wait for "Ready" under the second cell (about 30 seconds: it reads 9,659 apps and 37,427 reviews).
3. In the **Check an idea** cell, type your idea and pick the options in the form on the right, then run that cell
   again. Run it as often as you like; the model is only loaded once.

In [ ]:
#@title Get the project and install what Colab lacks
import os, sys, zipfile, subprocess

def find_root():
    for root in ["AppScout", ".", ".."]:
        if os.path.exists(os.path.join(root, "notebooks", "engine.py")):
            return os.path.abspath(root)

if find_root() is None:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Johnchrispindurai/appscout.git", "AppScout"], check=False)
if find_root() is None:
    if os.path.exists("AppScout.zip"):
        zipfile.ZipFile("AppScout.zip").extractall(".")
    else:
        from google.colab import files
        print("Upload AppScout.zip, the project zip.")
        uploaded = files.upload()
        zipfile.ZipFile(next(iter(uploaded))).extractall(".")
ROOT = find_root()
assert ROOT, "Could not find the project. Upload AppScout.zip and run this cell again."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-play-scraper", "altair>=5"], check=False)
print("Project at", ROOT)

In [ ]:
#@title Load the data and fit the model (once)
import json
import pandas as pd
from IPython.display import HTML, Markdown, display

sys.path[:0] = [os.path.join(ROOT, "notebooks"), os.path.join(ROOT, "app")]
import common as C
import engine as E
import views as V

RESULTS = json.loads(C.RESULTS.read_text(encoding="utf-8"))
eng = E.fit(C.load_apps(), C.load_reviews())
display(HTML(V.CSS))
print(f"Ready: {len(eng.apps):,} apps, {RESULTS['data']['reviews']['rows_with_text']:,} reviews.")

In [ ]:
#@title Check an idea
idea = "recipe app that plans meals from what is in your fridge"  #@param {type:"string"}
category = "Detect from my idea"  #@param ["Detect from my idea", "ART_AND_DESIGN", "AUTO_AND_VEHICLES", "BEAUTY", "BOOKS_AND_REFERENCE", "BUSINESS", "COMICS", "COMMUNICATION", "DATING", "EDUCATION", "ENTERTAINMENT", "EVENTS", "FAMILY", "FINANCE", "FOOD_AND_DRINK", "GAME", "HEALTH_AND_FITNESS", "HOUSE_AND_HOME", "LIBRARIES_AND_DEMO", "LIFESTYLE", "MAPS_AND_NAVIGATION", "MEDICAL", "NEWS_AND_MAGAZINES", "PARENTING", "PERSONALIZATION", "PHOTOGRAPHY", "PRODUCTIVITY", "SHOPPING", "SOCIAL", "SPORTS", "TOOLS", "TRAVEL_AND_LOCAL", "VIDEO_PLAYERS", "WEATHER"]
business_model = "Free"  #@param ["Free", "Paid"]
price_usd = 2.99  #@param {type:"number"}
content_rating = "Everyone"  #@param ["Everyone", "Everyone 10+", "Teen", "Mature 17+"]
search_play_store_today = True  #@param {type:"boolean"}

def section(title):
    display(HTML(f"<h2 style='font-family:Bricolage Grotesque,sans-serif;color:{V.INK};margin:1.6rem 0 .2rem'>{V.esc(title)}</h2>"))

def chart(title, note, c, width=560):
    display(HTML(V.CSS + V.title_html(title, note)))
    display(c.properties(width=width))

if not E.normalise(idea):
    display(Markdown("**Describe what the app does**, for example *meal planner for students*. "
                     "Words like app, free or best alone say nothing about the market."))
else:
    paid = business_model == "Paid"
    r = eng.evaluate(idea, category=None if category.startswith("Detect") else category, paid=paid,
                     price=price_usd if paid else 0.0, content=content_rating)
    display(HTML(V.CSS + V.listing_html(r, idea, content_rating, RESULTS)))
    if r["match_quality"] < 0.25:
        display(Markdown("*Few existing apps have names like this idea, so the comparison is weak. "
                         "Try the words a user would type into the store's search box.*"))

    section("Why this score")
    chart("What moved the score", "Each bar is one input's exact share of the answer, measured against an average app (log-odds).",
          V.drivers_chart(r))
    display(Markdown(V.how_judged_md(r)))

    section("Make it better")
    display(HTML(V.CSS + V.tips_html(V.tips(r, paid, RESULTS))))
    display(HTML(V.CSS + V.title_html("Better-aligned niches", "Genres close to your idea, ranked by closeness times how often apps there reach a million installs.")))
    display(V.niches_table(r).style.hide(axis="index"))

    section("Competitors")
    display(HTML(V.CSS + V.title_html("Closest apps in the 2018 data", "The 10 names most similar to your idea, with what they achieved.")))
    display(V.competitors_table(r).style.hide(axis="index"))
    if search_play_store_today:
        now = V.live_competitors(idea)
        display(HTML(V.CSS + V.title_html("On the Play Store today", "Top search results for your idea's words, fetched live.")))
        display(now.style.hide(axis="index") if now is not None else
                Markdown("*The Play Store could not be reached, so live results are left out.*"))

    section(f"The {V.nice(r['category'])} market")
    for title, note, c in V.market_charts(eng, r):
        chart(title, note, c)

    display(Markdown("**About the data and its limits**"))
    display(Markdown(V.data_limits_md(RESULTS)))

## Optional: rerun the whole analysis

The seven analysis notebooks in `notebooks/` produce every number above (they write `results/results.json`).
Running the next cell rebuilds them all; it takes about 15 minutes.

In [ ]:
# !cd "{ROOT}/notebooks" && python build.py